# Physical multilevel MOT trajectory animation

Configure one deterministic Section-12 trajectory, animate its path and all 24 state populations, and optionally save both as GIFs.

In [ ]:
%matplotlib inline
from dataclasses import replace
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, display

from pmot.magnetic_fields import default_anti_helmholtz_config
from pmot.mot_multilevel import (
    RateEquationAtomState, RateEquationTrajectoryConfig,
    build_multilevel_mot_beams, build_rate_equation_model,
    create_population_histogram_animation, create_trajectory_animation,
    default_multilevel_mot_config,
    multilevel_mot_paths, simulate_rate_equation_trajectory, trajectory_summary,
)
MODEL = build_rate_equation_model()
OUTPUT = multilevel_mot_paths()['figures'] / 'trajectory_animations'

In [ ]:
position = widgets.Text(description='r0 [mm]', value='8, 1, 0')
velocity = widgets.Text(description='v0 [m/s]', value='-3, 0.1, 0')
duration = widgets.FloatSlider(description='T [ms]', min=.1, max=30, step=.1, value=5, continuous_update=False)
dt = widgets.FloatSlider(description='dt [us]', min=1, max=20, step=1, value=5, continuous_update=False)
gradient = widgets.FloatSlider(description='dBz/dz [G/cm]', min=1, max=30, step=.5, value=10, continuous_update=False)
detuning = widgets.FloatSlider(description='cool Δ [MHz]', min=-40, max=-1, step=.5, value=-15, continuous_update=False)
power = widgets.FloatSlider(description='cool P [mW]', min=.1, max=60, step=.1, value=27, continuous_update=False)
repump_power = widgets.FloatSlider(description='repump P [mW]', min=.01, max=2, step=.01, value=.1, continuous_update=False)
gravity = widgets.Checkbox(description='gravity', value=True, indent=False)
frames = widgets.IntSlider(description='frames', min=20, max=400, step=10, value=180, continuous_update=False)
fps = widgets.IntSlider(description='fps', min=5, max=40, step=1, value=20, continuous_update=False)
save_gif = widgets.Checkbox(description='save GIF', value=False, indent=False)
run = widgets.Button(description='Animate trajectory', button_style='primary')
out = widgets.Output()

def parse_vector(text, scale=1.0):
    values = tuple(scale*float(value.strip()) for value in text.split(','))
    if len(values) != 3: raise ValueError('enter exactly three comma-separated values')
    return values

def animate(_=None):
    with out:
        out.clear_output(wait=True); plt.close('all')
        config = replace(
            default_multilevel_mot_config(),
            cooling_detuning_rad_per_s=2*np.pi*detuning.value*1e6,
            cooling_power_w_per_beam=power.value*1e-3,
            repump_power_w_per_beam=repump_power.value*1e-3,
            include_gravity=gravity.value,
        )
        coil = default_anti_helmholtz_config(target_gradient_g_per_cm=gradient.value)
        beams = build_multilevel_mot_beams(config=config)
        record = simulate_rate_equation_trajectory(
            RateEquationAtomState(parse_vector(position.value, 1e-3), parse_vector(velocity.value)),
            duration.value*1e-3, coil, beams=beams, model=MODEL, config=config,
            trajectory_config=RateEquationTrajectoryConfig(time_step_s=dt.value*1e-6),
        )
        path = None
        population_path = None
        if save_gif.value:
            stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            path = OUTPUT / f'trajectory_{stamp}.gif'
            population_path = OUTPUT / f'populations_24_state_{stamp}.gif'
        movie = create_trajectory_animation(record, beams, path, max_frames=frames.value, fps=fps.value)
        population_movie = create_population_histogram_animation(
            record, MODEL, population_path, max_frames=frames.value, fps=fps.value,
        )
        display(HTML(movie.to_jshtml()))
        display(HTML(population_movie.to_jshtml()))
        print(trajectory_summary(record))
        if path is not None: print('Saved:', path)
        if population_path is not None: print('Saved:', population_path)
        plt.close(movie._fig)
        plt.close(population_movie._fig)

run.on_click(animate)
display(widgets.VBox([
    widgets.HBox([position, velocity]), widgets.HBox([duration, dt, gradient]),
    widgets.HBox([detuning, power, repump_power, gravity]),
    widgets.HBox([frames, fps, save_gif, run]), out,
]))